# Artificial Intelligence
# 464
# Quiz #5

## Before You Begin...
00. We're using a Jupyter Notebook environment (tutorial available here: https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html),
01. Read the entire notebook before beginning your work, and
02.  Check the submission deadline on Gradescope.


## General Directions for this Assignment
00. Output format should be exactly as requested (it is your responsibility to make sure notebook looks as expected on Gradescope), and
01. Functions should do only one thing.


## Before You Submit...
00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
03.  Do not submit any other files.

## Language Modeling

This homework will require you to load and train models.  If you choose small models and datasets, you should be able to run this locally on your computer. However, larger models/datasets may require GPU access. You can access one GPU for free on [Google Colab](https://colab.research.google.com/).

We will use HuggingFace libraries in this quiz. We discussed majority of what you will need during the discussion demo. Additional documentation can be found [here](https://huggingface.co/docs).

In [1]:

# Imports
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import pipeline
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM
import os
os.environ["WANDB_DISABLED"] = "true"

## Problem 0: Data
From the [HuggingFace Datasets](https://huggingface.co/datasets), choose a dataset that satisfies the following criteria:
- Data must have train and test splits (Optional development set)
- Task must be text classification
- Task must have at least 3 labels


In [2]:
# Load the data here
from datasets import load_dataset

dataset = load_dataset("CogComp/trec")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'coarse_label', 'fine_label'],
        num_rows: 5452
    })
    test: Dataset({
        features: ['text', 'coarse_label', 'fine_label'],
        num_rows: 500
    })
})


**Describe the data.**
What is the utility of the task? What are the inputs? What are the labels? Are the any potential difficulties you expect from the task? How do you evaluate the performance of this task?

The AG News dataset is a popular benchmark for classifying news articles into four categories: World, Sports, Business, and Sci/Tech. The dataset includes both titles and descriptions as inputs, making it a great test case for models that process short-form text. This kind of classification is useful for things like news aggregation, content moderation, and search engine organization, helping users quickly find relevant articles. However, the task isn't without challenges. Some news headlines can be ambiguous, certain categories may have more data than others (causing class imbalance), and since titles are usually short, they might not have enough context for an accurate classification. Additionally, industry-specific jargon in business or tech news could make classification trickier. To measure performance, we typically use metrics like accuracy, precision, recall, and F1-score. A confusion matrix also helps by showing which categories are most often confused with each other. The goal is to build a model that not only gets most classifications right but also performs well across all four categories.

**Research current methods using this dataset.**
What is the current state of the art method? Describe the method, including the type of model used, training protocol (if any), and the performance. Cite your sources.

The AG News dataset is a really good benchmark for text classification, especially for sorting news into categories. Over the years, transformer-based models have significantly boosted accuracy on this dataset. One major breakthrough came in 2019 when XLNet, a powerful autoregressive pretraining model, set a new record with an error rate of just 4.45%. XLNet is unique because it combines the best of autoregressive (predicting the next word) and autoencoding (understanding whole sentence structures) models. Instead of reading text in a fixed order, it learns from all possible word permutations, making it great at capturing complex relationships in news articles. Training XLNet happens in two stages: pretraining and fine-tuning. XLNet isn’t the only transformer making waves. BERT, another well-known model, has also been fine-tuned for AG News. One study got 94.22% accuracy with a standard BERT classifier, while another pushed the limits further by combining BERT with a CNN-enhanced Transformer-Encoder, reaching an impressive 98.9% accuracy, the best reported so far. These transformer-based models have completely changed the game for news categorization and text classification, proving that understanding context is key to making accurate predictions.

Sources:

https://github.com/Daammon/AG-News-Classifier

https://arxiv.org/abs/2209.06344?utm

https://paperswithcode.com/sota/text-classification-on-ag-news

(Optional) If necessary, perform any data preprocessing here. For example, depending on the dataset you choose, you may need to clean the text or split the training set into a train and validation set.

In [3]:
# TODO: Dataset preprocessing (Optional)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
def preprocess_function(examples):
    result = tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    result["labels"] = examples["coarse_label"]
    return result
tokenized_datasets = dataset.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text","fine_label","coarse_label"])
tokenized_datasets.set_format("torch")


## Problem 1: encoder-only models or decoder-only models
## Option A: encoder-only models
Choose an encoder-only model (e.g. BERT). Load the model and add a classification layer.

Describe the model you choose. What are the unique properties of this model? What are the pros and cons? Cite your sources.

TinyBERT is a lightweight version of BERT, designed to be faster and more efficient while still delivering solid performance. The specific variant, "huawei-noah/TinyBERT_General_4L_312D", is much smaller than standard BERT, making it ideal for resource-limited environments like mobile devices and edge computing. TinyBERT achieves this efficiency through knowledge distillation, meaning it learns from a larger, pre-trained BERT model by mimicking its behavior at different layers. With just four transformer layers and a hidden size of 312, it significantly reduces memory and power usage compared to full-sized BERT models. While TinyBERT performs well across many natural language processing (NLP) tasks, there are trade-offs. Its smaller size limits its ability to capture complex contextual relationships, which can be a drawback for tasks requiring deep semantic understanding. Also, getting the most out of TinyBERT requires fine-tuning and task-specific distillation, which can be computationally demanding. Despite these challenges, TinyBERT is a great choice for scenarios where speed, scalability, and low computational cost matter more than absolute accuracy.

Sources:

https://arxiv.org/abs/1909.10351

https://arxiv.org/abs/1810.04805

Finetune the model on your dataset. Report the performance on the test set.

In [4]:
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from torch.optim import AdamW
from transformers import get_scheduler, AutoModelForSequenceClassification

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = AutoModelForSequenceClassification.from_pretrained(
    "huawei-noah/TinyBERT_General_4L_312D", 
    num_labels=6
)

train_loader = DataLoader(tokenized_datasets["train"], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized_datasets["test"], batch_size=64)
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

num_training_steps = len(train_loader) * 3
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

criterion = torch.nn.CrossEntropyLoss()

num_epochs = 3
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    model.train()
    total_train_loss = 0
    train_progress_bar = tqdm(train_loader, desc="Training", leave=False)

    for batch in train_progress_bar:
        optimizer.zero_grad()

        inputs = {key: val.to(device) for key, val in batch.items() if key != "labels"}
        labels = batch["labels"].to(device)

        outputs = model(**inputs)
        logits = outputs.logits

        loss = criterion(logits, labels)
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        train_progress_bar.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Training loss: {avg_train_loss:.4f}")
    
    model.eval()
    total_val_loss = 0
    num_correct = 0
    num_samples = 0
    val_progress_bar = tqdm(val_loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in val_progress_bar:
            inputs = {key: val.to(device) for key, val in batch.items() if key != "labels"}
            labels = batch["labels"].to(device)

            outputs = model(**inputs)
            logits = outputs.logits

            loss = criterion(logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=-1)
            num_correct += (preds == labels).sum().item()
            num_samples += labels.size(0)
            
    avg_val_loss = total_val_loss / len(val_loader)
    accuracy = num_correct / num_samples
    print(f"Validation loss: {avg_val_loss:.4f}")
    print(f"Validation accuracy: {accuracy:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3


Training:   0%|          | 0/341 [00:00<?, ?it/s]

Training loss: 0.9873


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

...

Validation loss: 0.2872
Validation accuracy: 0.9280
Epoch 3/3


Training:   0%|          | 0/341 [00:00<?, ?it/s]

Training loss: 0.2665


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Validation loss: 0.2355
Validation accuracy: 0.9400


## Option B: decoder-only models
Choose an decoder-only model (e.g. GPT2). Describe the model you choose. What are the unique properties of this model? What are the pros and cons? Cite your sources.

TODO

Load the model and use prompting for your task. You will likely need to write a helper function to parse the answer.

(Ex. “The answer is 1” -> 1). Report the performance on the test set.

In [5]:
# TODO

## Problem 2: Error Analysis

Conduct an error analysis on your models. What are your models good at? What do they get wrong? Provide examples of both correct and incorrect predictions. Suggest methods to improve the performance.

In [6]:
import numpy as np
from sklearn.metrics import classification_report
import collections

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        inputs = {key: val.to(device) for key, val in batch.items() if key in ["input_ids", "attention_mask", "token_type_ids"]}
        labels = batch["labels"].to(device)
        outputs = model(**inputs)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print("Predicted Label Distribution:", collections.Counter(all_predictions))
print("Actual Label Distribution:", collections.Counter(all_labels))

target_names = dataset["train"].features["coarse_label"].names
report = classification_report(all_labels, all_predictions, target_names=target_names, digits=4, zero_division=1)
print("Classification Report:")
print(report)


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Predicted Label Distribution: Counter({2: 151, 5: 113, 1: 86, 4: 84, 3: 65, 0: 1})
Actual Label Distribution: Counter({2: 138, 5: 113, 1: 94, 4: 81, 3: 65, 0: 9})
Classification Report:
              precision    recall  f1-score   support

        ABBR     1.0000    0.1111    0.2000         9
        ENTY     0.9535    0.8723    0.9111        94
        DESC     0.9007    0.9855    0.9412       138
         HUM     0.9692    0.9692    0.9692        65
         LOC     0.9286    0.9630    0.9455        81
         NUM     0.9735    0.9735    0.9735       113

    accuracy                         0.9400       500
   macro avg     0.9542    0.8124    0.8234       500
weighted avg     0.9423    0.9400    0.9338       500



## OPTIONAL. BONUS. Problem 3: Improvements

Implement your suggestions for improving the performance. Describe your method and report the results on the test set.

In [7]:
# TODO

No other directions for this quiz, other than what's here and in the "General Directions" section. You have a lot of freedom with this quiz. Don't get carried away. It is expected the results may vary, being better or worse. Graders are not going to run your notebooks. The notebook will be read as a report on how different models were explored. Since you'll be using libraries, the emphasis will be on your ability to communicate your findings.

## Before You Submit...

00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
03.  Do not submit any other files.